In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os
import sys
# add the project root to the path so the src package can be imported
sys.path.append(os.path.abspath('..'))

from torch import load,device,tensor
import torch.nn.functional as F
from torchvision.datasets import ImageFolder
from torchvision.utils import make_grid

from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import yaml
from src.model_utils import get_transforms,denorm,FocalLoss,build_swin_model,predict_classifier
from src.train_classifier import train
from src.vis import show_curves,show_confusion_matrix


In [2]:
with open('../configs/stage3.yaml','r') as f:

    stage3_config = yaml.safe_load(f)

with open(stage3_config['paths']['data_yaml_path'],'r') as f:

    dis_data_yaml = yaml.safe_load(f)

with open('../configs/trained_models.yaml','r') as f:

    trained_models_config = yaml.safe_load(f)

In [3]:
stage3_config

{'paths': {'original_images_path': '../Data/Raw/DENTEX CHALLENGE 2023/Training_data/quadrant-enumeration-disease/xrays',
  'original_json_path': '../Data/Raw/DENTEX CHALLENGE 2023/Training_data/quadrant-enumeration-disease/train_quadrant_enumeration_disease.json',
  's3_main_path': '../Data/Processed/Stage 3 (Disease Classifier)',
  'runs_s3_output': '../Runs/Stage 3',
  'data_yaml_path': '../Data/Processed/Stage 3 (Disease Classifier)/data.yaml'},
 'healthy_unhealthy': {'model_args': {'image_size': 256,
   'epochs': 50,
   'batch_size': 64,
   'patience': 10,
   'lr': 0.0001,
   'num_workers': 4,
   'device_name': 'cuda',
   'save_dir': '../Runs/Stage 3/healthy_unhealthy'}},
 'disease_classifier': {'model_args': {'image_size': 256,
   'epochs': 50,
   'batch_size': 64,
   'patience': 10,
   'lr': 0.001,
   'num_workers': 4,
   'device_name': 'cuda',
   'save_dir': '../Runs/Stage 3/disease_classifier'}},
 'caries_or_deepcaries': {'model_args': {'image_size': 256,
   'epochs': 50,
   'b

In [4]:
dis_data_yaml

{'healthy_unhealthy': {'train': '../Data/Processed/Stage 3 (Disease Classifier)/healthy or un-healthy classifier/train/',
  'val': '../Data/Processed/Stage 3 (Disease Classifier)/healthy or un-healthy classifier/valid/',
  'test': '../Data/Processed/Stage 3 (Disease Classifier)/healthy or un-healthy classifier/test/',
  'nc': 2,
  'names': ['Disease Found', 'Healthy']},
 'disease_classifier': {'train': '../Data/Processed/Stage 3 (Disease Classifier)/disease classifier/train/',
  'val': '../Data/Processed/Stage 3 (Disease Classifier)/disease classifier/valid/',
  'test': '../Data/Processed/Stage 3 (Disease Classifier)/disease classifier/test/',
  'nc': 3,
  'names': ['Impacted', 'Caries', 'Periapical']},
 'caries_or_deepcaries': {'train': '../Data/Processed/Stage 3 (Disease Classifier)/caries or deep_caries classifier/train/',
  'val': '../Data/Processed/Stage 3 (Disease Classifier)/caries or deep_caries classifier/valid/',
  'test': '../Data/Processed/Stage 3 (Disease Classifier)/carie

In [5]:
trained_models_config

{'quadrant_detection_model': {'best': '../Runs/Stage 1/weights/best.pt',
  'last': '../Runs/Stage 1/weights/last.pt'},
 'enumeration_detection_model': {'best': '../Runs/Stage 2/weights/best.pt',
  'last': '../Runs/Stage 2/weights/last.pt'},
 'enumeration_continued_detection_model': {'best': '../Runs/Stage 2 Continued/weights/best.pt',
  'last': '../Runs/Stage 2 Continued/weights/last.pt'},
 'healthy_unhealthy_model': {'best': '../Runs/Stage 3/Healthy & Un-Healthy Classifier/weights/best.pt',
  'last': '../Runs/Stage 3/Healthy & Un-Healthy Classifier/weights/last.pt'},
 'disease_classification_model': {'best': '../Runs/Stage 3/Disease Classifier/weights/best.pt',
  'last': '../Runs/Stage 3/Disease Classifier/weights/last.pt'},
 'caries_deepcaries_model': {'best': '../Runs/Stage 3/Caries & Deep Caries Classifier/weights/best.pt',
  'last': '../Runs/Stage 3/Caries & Deep Caries Classifier/weights/last.pt'}}

In [6]:
H_UH_CONFIG = stage3_config['healthy_unhealthy']['model_args']
DIS_CONFIG = stage3_config['disease_classifier']['model_args']
C_DC_CONFIG = stage3_config['caries_or_deepcaries']['model_args']

In [7]:
h_uh_train_dataset = ImageFolder(dis_data_yaml['healthy_unhealthy']['train'], transform = get_transforms(H_UH_CONFIG['image_size'], True))
h_uh_valid_dataset = ImageFolder(dis_data_yaml['healthy_unhealthy']['val']  , transform = get_transforms(H_UH_CONFIG['image_size']))
h_uh_test_dataset  = ImageFolder(dis_data_yaml['healthy_unhealthy']['test'] , transform = get_transforms(H_UH_CONFIG['image_size']))

h_uh_train_dl = DataLoader(h_uh_train_dataset, batch_size=H_UH_CONFIG['batch_size'], shuffle=True , num_workers=H_UH_CONFIG['num_workers'], pin_memory=True)
h_uh_valid_dl = DataLoader(h_uh_valid_dataset, batch_size=H_UH_CONFIG['batch_size'], shuffle=False, num_workers=H_UH_CONFIG['num_workers'], pin_memory=True)
h_uh_test_dl  = DataLoader(h_uh_test_dataset , batch_size=H_UH_CONFIG['batch_size'], shuffle=False, num_workers=H_UH_CONFIG['num_workers'], pin_memory=True)
print(h_uh_train_dataset.classes)

['1_disease found', '2_no disease found']


In [ ]:
dis_train_dataset = ImageFolder(dis_data_yaml['disease_classifier']['train'], transform = get_transforms(DIS_CONFIG['image_size'],True))
dis_valid_dataset = ImageFolder(dis_data_yaml['disease_classifier']['val']  , transform = get_transforms(DIS_CONFIG['image_size']))
dis_test_dataset  = ImageFolder(dis_data_yaml['disease_classifier']['test'] , transform = get_transforms(DIS_CONFIG['image_size']))

dis_train_dl = DataLoader(dis_train_dataset, batch_size=DIS_CONFIG['batch_size'], shuffle=True , num_workers=DIS_CONFIG['num_workers'], pin_memory=True)
dis_valid_dl = DataLoader(dis_valid_dataset, batch_size=DIS_CONFIG['batch_size'], shuffle=False, num_workers=DIS_CONFIG['num_workers'], pin_memory=True)
dis_test_dl  = DataLoader(dis_test_dataset , batch_size=DIS_CONFIG['batch_size'], shuffle=False, num_workers=DIS_CONFIG['num_workers'], pin_memory=True)
print(dis_train_dataset.classes)

In [ ]:
c_dc_train_dataset = ImageFolder(dis_data_yaml['caries_or_deepcaries']['train'], transform = get_transforms(C_DC_CONFIG['image_size'],True))
c_dc_valid_dataset = ImageFolder(dis_data_yaml['caries_or_deepcaries']['val']  , transform = get_transforms(C_DC_CONFIG['image_size']))
c_dc_test_dataset  = ImageFolder(dis_data_yaml['caries_or_deepcaries']['test'] , transform = get_transforms(C_DC_CONFIG['image_size']))

c_dc_train_dl = DataLoader(c_dc_train_dataset, batch_size=C_DC_CONFIG['batch_size'], shuffle=True , num_workers=C_DC_CONFIG['num_workers'], pin_memory=True)
c_dc_valid_dl = DataLoader(c_dc_valid_dataset, batch_size=C_DC_CONFIG['batch_size'], shuffle=False, num_workers=C_DC_CONFIG['num_workers'], pin_memory=True)
c_dc_test_dl  = DataLoader(c_dc_test_dataset , batch_size=C_DC_CONFIG['batch_size'], shuffle=False, num_workers=C_DC_CONFIG['num_workers'], pin_memory=True)
print(c_dc_train_dataset.classes)

In [ ]:
plt.figure(figsize=(16,10))
plt.imshow(make_grid(denorm(next(iter(h_uh_train_dl))[0])).permute(1,2,0))

In [ ]:
plt.figure(figsize=(16,10))
plt.imshow(make_grid(denorm(next(iter(dis_train_dl))[0])).permute(1,2,0))

In [ ]:
plt.figure(figsize=(16,10))
plt.imshow(make_grid(denorm(next(iter(c_dc_train_dl))[0])).permute(1,2,0))